[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/29_cnn/02_cnn_with_pytorch.ipynb)

# Bài 02 — CNN với PyTorch

**🎯 Mục tiêu**
- Xây CNN bằng `nn.Conv2d`, `nn.MaxPool2d`, `nn.AdaptiveAvgPool2d`; theo dõi shape qua từng lớp.
- So sánh với MLP: độ chính xác và số tham số.
- **Thiên kiến quy nạp**: train ở nửa trái ảnh, test ở nửa phải — MLP vs CNN có global pooling vs CNN "flatten".
- Data augmentation; trực quan hoá bộ lọc và bản đồ đặc trưng (dữ liệu thật: bài 04).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/29_cnn

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.datasets import SHAPE_CLASSES, make_shapes
from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
device = "cpu"                                                          # mô hình nhỏ: CPU đủ nhanh và kết quả tái lập; đổi "cuda" nếu có GPU


def to_tensor(X):
    return torch.tensor(X[:, None], dtype=torch.float32)                # (N, 1, 28, 28): thêm chiều kênh


X_train, y_train = make_shapes(3000, seed=0)
X_test, y_test = make_shapes(1000, seed=1)
Xtr, Xte = to_tensor(X_train), to_tensor(X_test)
ytr, yte = torch.tensor(y_train), torch.tensor(y_test)
print("train:", tuple(Xtr.shape), "| test:", tuple(Xte.shape), "| lớp:", SHAPE_CLASSES)

## 1. Một CNN nhỏ và shape qua từng lớp

`Conv(3×3, padding 1)` giữ kích thước; `MaxPool(2)` chia đôi. Cuối mạng: **global average pooling** (trung bình mỗi bản đồ đặc trưng) → lớp tuyến tính.

In [ ]:
def make_cnn(head="gap"):
    features = [nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(32, 64, 3, padding=1), nn.ReLU()]
    if head == "gap":
        classifier = [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 5)]
    else:                                                               # "flatten": giữ vị trí không gian
        classifier = [nn.Flatten(), nn.Linear(64 * 7 * 7, 5)]
    return nn.Sequential(*features, *classifier)


def make_mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 5))


x = Xtr[:2]
for layer in make_cnn("gap"):
    x = layer(x)
    print(f"{layer.__class__.__name__:18s} → {tuple(x.shape)}")


def n_params(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
def train(model, X, y, epochs=15, lr=1e-3, seed=0, augment=None):
    torch.manual_seed(seed)
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(X), generator=g)
        for s in range(0, len(perm), 64):
            idx = perm[s:s + 64]
            xb = X[idx] if augment is None else augment(X[idx], g)
            opt.zero_grad()
            nn.functional.cross_entropy(model(xb.to(device)), y[idx].to(device)).backward()
            opt.step()
    return model


def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X.to(device)).argmax(1).cpu() == y).float().mean().item()

## 2. CNN vs MLP

In [ ]:
results = {}
for name, make in [("MLP 784-256-128-5", make_mlp), ("CNN + global avg pool", lambda: make_cnn("gap")), ("CNN + flatten", lambda: make_cnn("flatten"))]:
    torch.manual_seed(0)
    model = make()
    t0 = time.perf_counter()
    train(model, Xtr, ytr)
    results[name] = model
    print(f"{name:24s}: {n_params(model):7,d} tham số | accuracy test = {accuracy(model, Xte, yte):.3f} | {time.perf_counter() - t0:.1f}s")

CNN chính xác hơn hẳn với **ít tham số hơn 6–10 lần**: chia sẻ trọng số giúp mỗi bộ dò "góc", "cạnh cong", "đầu mút" được học **một lần** và dùng ở mọi vị trí; MLP phải học lại cho từng vị trí với lượng dữ liệu hạn chế.

## 3. Thiên kiến quy nạp: train bên trái, test bên phải

Train **chỉ** với hình nằm ở nửa **trái** ảnh; test với hình ở nửa **phải** (chưa từng thấy vị trí đó):

In [ ]:
X_left, y_left = make_shapes(3000, region="left", seed=2)
X_left_te, y_left_te = make_shapes(1000, region="left", seed=4)
X_right_te, y_right_te = make_shapes(1000, region="right", seed=3)
XL, XLt, XRt = to_tensor(X_left), to_tensor(X_left_te), to_tensor(X_right_te)
yL, yLt, yRt = torch.tensor(y_left), torch.tensor(y_left_te), torch.tensor(y_right_te)
shift_results = {}
for name, make in [("MLP", make_mlp), ("CNN + global avg pool", lambda: make_cnn("gap")), ("CNN + flatten", lambda: make_cnn("flatten"))]:
    torch.manual_seed(0)
    model = train(make(), XL, yL)
    shift_results[name] = (accuracy(model, XLt, yLt), accuracy(model, XRt, yRt))
    print(f"{name:24s}: test nửa trái (cùng vùng) = {shift_results[name][0]:.3f} | test nửa phải (vùng mới) = {shift_results[name][1]:.3f}")

- **MLP**: ở vùng mới gần như **đoán mò** (5 lớp → 0.2) — mỗi trọng số gắn với một pixel cố định.
- **CNN + global average pooling**: không giảm — tích chập **đồng biến**, GAP lấy trung bình toàn ảnh nên **bất biến** với vị trí.
- **CNN + flatten**: tích chập vẫn tạo đặc trưng tốt, nhưng lớp `Linear` cuối gắn trọng số với **vị trí** trong bản đồ 7×7 → mất một phần bất biến.

Lựa chọn kiến trúc = lựa chọn **giả định** về dữ liệu. Giả định đúng → học được với ít dữ liệu và tổng quát hoá ra ngoài phân phối train.

## 4. Data augmentation

Nếu kiến trúc chưa bất biến (CNN + flatten), có thể **dạy** bất biến bằng dữ liệu: dịch ngẫu nhiên ảnh khi train.

In [ ]:
def random_shift(xb, g, max_shift=8):
    out = torch.empty_like(xb)
    shifts = torch.randint(-max_shift, max_shift + 1, (len(xb), 2), generator=g)
    for i, (dy, dx) in enumerate(shifts.tolist()):
        out[i] = torch.roll(xb[i], shifts=(dy, dx), dims=(1, 2))         # ảnh nền gần 0 nên cuộn vòng chấp nhận được
    return out


torch.manual_seed(0)
aug_model = train(make_cnn("flatten"), XL, yL, augment=random_shift)
acc_l, acc_r = accuracy(aug_model, XLt, yLt), accuracy(aug_model, XRt, yRt)
print(f"CNN + flatten + dịch ngẫu nhiên: nửa trái = {acc_l:.3f} | nửa phải = {acc_r:.3f} (không augmentation: {shift_results['CNN + flatten'][1]:.3f})")

Augmentation lấp phần lớn khoảng cách. Với ảnh thật: lật ngang, cắt ngẫu nhiên, xoay nhỏ, đổi độ sáng/màu (`torchvision.transforms`) — chỉ dùng phép biến đổi **giữ nguyên nhãn** (lật dọc chữ số "6" thành "9" là sai!).

## 5. Mạng đã học gì?

In [ ]:
cnn = results["CNN + global avg pool"]
filters = cnn[0].weight.detach().cpu().numpy()[:, 0]                    # 16 bộ lọc 3×3 của lớp đầu
fig, axes = plt.subplots(2, 8, figsize=(14, 3.8))
for ax, f in zip(axes.ravel(), filters):
    ax.imshow(f, cmap="RdBu_r", vmin=-np.abs(filters).max(), vmax=np.abs(filters).max())
    ax.axis("off")
fig.suptitle("16 bộ lọc 3×3 lớp đầu (đỏ dương, xanh âm)")
plt.show()

sample = Xte[np.flatnonzero(y_test == 3)[0]][None]                       # một chữ thập
with torch.no_grad():
    fmap1 = cnn[1](cnn[0](sample))[0]                                    # sau conv1 + ReLU
    fmap3 = cnn[:8](sample)[0]                                           # sau conv3 + ReLU (7×7)
fig, axes = plt.subplots(2, 9, figsize=(18, 4.5))
axes[0, 0].imshow(sample[0, 0], cmap="gray")
axes[0, 0].set_title("ảnh")
for j in range(8):
    axes[0, j + 1].imshow(fmap1[j], cmap="viridis")
    axes[1, j + 1].imshow(fmap3[j], cmap="viridis")
axes[1, 0].axis("off")
axes[0, 1].set_title("bản đồ đặc trưng lớp 1 (28×28)", loc="left", fontsize=9)
axes[1, 1].set_title("lớp 3 (7×7) — trừu tượng hơn", loc="left", fontsize=9)
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

Lớp đầu học các bộ dò **cạnh/hướng** (giống Sobel ở bài 01); lớp sâu hơn có trường tiếp nhận lớn hơn, phản ứng với **tổ hợp** cạnh (góc, đầu mút) — phân cấp đặc trưng từ đơn giản đến phức tạp (Zeiler & Fergus, 2014).

## 6. Tiếp theo: ảnh thật

Bộ hình học tổng hợp giúp thấy rõ từng hiện tượng. [Bài 04](04_cnn_on_real_images.ipynb) áp dụng đúng các ý tưởng này trên hai bộ ảnh thật — **Fashion-MNIST** và **CIFAR-10** — và so sánh trực tiếp với mô hình tuyến tính (chương 07) và MLP (chương 27).

## 🧠 Kiểm tra nhanh

1. Vì sao CNN cần ít tham số hơn MLP mà lại chính xác hơn trên ảnh?
2. Giải thích kết quả train-trái/test-phải của ba mô hình.
3. Global average pooling khác flatten ở điểm nào? Ưu nhược?
4. Augmentation nào hợp lệ cho ảnh chữ số? Cho ảnh y tế X-quang?
5. Lớp đầu và lớp sâu của CNN thường học loại đặc trưng nào?

## 📌 Tóm tắt

- CNN = Conv + ReLU + Pool lặp lại + đầu phân loại; ít tham số, chính xác hơn MLP trên ảnh.
- Conv đồng biến, global pooling bất biến → tổng quát hoá ra vị trí mới; flatten mất một phần bất biến; augmentation dạy bất biến bằng dữ liệu.
- Đặc trưng phân cấp: cạnh → tổ hợp → đối tượng.

## 📚 Tài liệu tham khảo

- LeCun et al. (1998) — *Gradient-Based Learning Applied to Document Recognition*
- Zeiler & Fergus (2014) — *Visualizing and Understanding Convolutional Networks*
- Lin, Chen & Yan (2014) — *Network In Network* (global average pooling)
- Battaglia et al. (2018) — *Relational inductive biases, deep learning, and graph networks*